In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies = movies.merge(credits, on='title')

print(f"Loaded {len(movies)} movies.")
print(movies.columns.tolist())  

Loaded 4809 movies.
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'cast', 'crew']


In [4]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

def extract_names(text):
    """
    Converts a JSON string like '[{"name": "Action"}, {"name": "Drama"}]'
    into a list ['Action', 'Drama']
    """
    try:
        items = ast.literal_eval(text)  
        return [item['name'] for item in items]
    except:
        return []

def extract_top_cast(text, max_cast=3):
    """
    Same as above, but only keeps the first 3 cast members.
    More cast = more noise for our algorithm.
    """
    try:
        items = ast.literal_eval(text)
        return [item['name'] for item in items[:max_cast]]
    except:
        return []

def extract_director(text):
    """
    The 'crew' column has everyone: director, producer, etc.
    We specifically want the person whose job is 'Director'.
    """
    try:
        items = ast.literal_eval(text)
        for item in items:
            if item['job'] == 'Director':
                return [item['name']]
        return []
    except:
        return []


movies['genres']   = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast']     = movies['cast'].apply(extract_top_cast)
movies['crew']     = movies['crew'].apply(extract_director)

movies.dropna(subset=['overview'], inplace=True)

In [ ]:
def collapse_spaces(words):
    """Joins a list of words, removing internal spaces."""
    return [w.replace(" ", "") for w in words]

movies['tags'] = (
    movies['overview'].apply(lambda x: x.split())   # Split overview into words
    + movies['genres'].apply(collapse_spaces)
    + movies['keywords'].apply(collapse_spaces)
    + movies['cast'].apply(collapse_spaces)
    + movies['crew'].apply(collapse_spaces)
)

# Convert the list of words back into a single string
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

# Keep only the columns we need going forward
final = movies[['movie_id', 'title', 'tags']].reset_index(drop=True)

print("\nSample tag for 'Avatar':")
print(final[final['title'] == 'Avatar']['tags'].values[0][:300])


Sample tag for 'Avatar':
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance spac


In [6]:
print(final[final['title'] == 'Avatar']['tags'].values[0][:])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron


In [7]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

tfidf_matrix = vectorizer.fit_transform(final['tags'])

print(f"\nTF-IDF matrix shape: {tfidf_matrix.shape}")


TF-IDF matrix shape: (4806, 5000)


In [9]:
similarity = cosine_similarity(tfidf_matrix)

print(f"\nSimilarity matrix shape: {similarity.shape}")


Similarity matrix shape: (4806, 4806)


In [10]:
def recommend(movie_title, num_recommendations=10):
    """
    Given a movie title, returns a list of similar movies.
    
    How it works:
    1. Find the row index of the given movie
    2. Look up that row in the similarity matrix
    3. Sort all movies by similarity score (highest first)
    4. Return the top N (skipping index 0, which is the movie itself)
    """
    
    # Find the movie in our dataframe (case-insensitive search)
    matches = final[final['title'].str.lower() == movie_title.lower()]
    
    if matches.empty:
        print(f"❌ Movie '{movie_title}' not found!")
        # Suggest close matches
        close = final[final['title'].str.lower().str.contains(movie_title.lower())]
        if not close.empty:
            print("Did you mean one of these?")
            for t in close['title'].head(5):
                print(f"  - {t}")
        return []
    
    # Get the integer position (iloc index) of this movie
    idx = matches.index[0]
    
    # Get this movie's row from the similarity matrix
    # Result: a list of (movie_index, similarity_score) pairs
    sim_scores = list(enumerate(similarity[idx]))
    
    # Sort by similarity score, highest first
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first result (that's the movie itself, similarity = 1.0)
    sim_scores = sim_scores[1 : num_recommendations + 1]
    
    # Get the actual movie titles
    movie_indices = [i[0] for i in sim_scores]
    recommendations = final['title'].iloc[movie_indices].tolist()
    
    return recommendations

In [11]:
test_movies = ['The Dark Knight', 'Avengers', 'Inception', 'Toy Story']

for movie in test_movies:
    print(f"\n🎬 Movies similar to '{movie}':")
    results = recommend(movie)
    for i, rec in enumerate(results, 1):
        print(f"  {i}. {rec}")


🎬 Movies similar to 'The Dark Knight':
  1. The Dark Knight Rises
  2. Batman Returns
  3. Batman Begins
  4. Batman Forever
  5. Batman: The Dark Knight Returns, Part 2
  6. Batman v Superman: Dawn of Justice
  7. Batman & Robin
  8. Batman
  9. Batman
  10. The Lego Movie

🎬 Movies similar to 'Avengers':
❌ Movie 'Avengers' not found!
Did you mean one of these?
  - Avengers: Age of Ultron
  - The Avengers

🎬 Movies similar to 'Inception':
  1. Don Jon
  2. Premium Rush
  3. Cypher
  4. Hesher
  5. Duplex
  6. The Walk
  7. Pitch Perfect 2
  8. (500) Days of Summer
  9. Mission: Impossible - Rogue Nation
  10. Looper

🎬 Movies similar to 'Toy Story':
  1. Toy Story 3
  2. Toy Story 2
  3. The 40 Year Old Virgin
  4. Factory Girl
  5. Class of 1984
  6. For Your Consideration
  7. Heartbeeps
  8. Man on the Moon
  9. Small Soldiers
  10. The 41–Year–Old Virgin Who Knocked Up Sarah Marshall and Felt Superbad About It


In [12]:
import pickle

with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)

final.to_csv('movies_clean.csv', index=False)

print("Saved similarity.pkl and movies_clean.csv ✅")

Saved similarity.pkl and movies_clean.csv ✅
